# Replication Package Notebook

## Contents

- [Pre-work](#pre-work)
- [Data Collection](#data-collection)
  - [Novielli et al., 2020](#novielli-et-al-2020)
    - [Test/Train split](#testtrain-split)
  - [Coutinho et al., 2024](#coutinho-et-al-2024)
  - [Romanian Gold Standard](#romanian-gold-standard)
    - [Label distribution](#label-distribution)
    - [Inter-annotator agreement (Fleiss kappa)](#inter-annotator-agreement-fleiss-kappa)
    - [Polarity by comment type](#polarity-by-comment-type)
    - [Chi-square test of independence](#chi-square-test-of-independence)
- [DEVA Experiments (English datasets)](#deva-experiments-english-datasets)
  - [Mapping DEVA results to polarity labels](#mapping-deva-results-to-polarity-labels)
  - [Calculating precision, recall, F1 for DEVA results](#calculating-precision-recall-f1-for-deva-results)
- [LLM Experiments](#llm-experiments)
  - [Label Parser](#label-parser)
  - [Prompting and Classification](#prompting-and-classification)
  - [Running LLMs](#running-llms)
  - [Romanian Gold Standard Results](#romanian-gold-standard-results)
  - [Prompt sensitivity](#prompt-sensitivity)
  - [Parser source and fallback rate](#parser-source-and-fallback-rate)
- [Export combined predictions for downstream analysis](#export-combined-predictions-for-downstream-analysis)

# Pre-work

In [1]:
# imports

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import ollama
import re as _re

from scipy import stats
from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
# config

tqdm.pandas()

In [3]:
# constants

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "datasets"

NOVIELLI_GOLD_STANDARD_DIR = DATASET_DIR / "novielli-gh-gold"
GH_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "github_gold.csv"

COUTINHO_DIR = DATASET_DIR / "coutinho-sentiment-replication"
COUTINHO = COUTINHO_DIR / "dataset.json"

RO_GOLD_DIR = DATASET_DIR / "ro-gold"
RO_GOLD = RO_GOLD_DIR / "ro_gold_final.json"

DEVA_INPUT_DIR = ROOT_DIR / "input" / "deva"
DEVA_OUTPUT_DIR = ROOT_DIR / "output" / "deva"

PRED_DIR = ROOT_DIR / "output" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# Data Collection

## Novielli et al., 2020
The dataset used by Novielli et al. is in CSV format, which makes it straight forward to create a dataframe out of.

In [4]:
df_gold_standard_full = pd.read_csv(GH_GOLD_STANDARD, sep=';', quotechar='"', encoding='utf-8')
df_gold_standard_full.columns = df_gold_standard_full.columns.str.lower()

# strip outer quote
df_gold_standard_full['text'] = df_gold_standard_full['text'].str.rstrip('"')

# remove id column
df_gold_standard_full = df_gold_standard_full[['text', 'polarity']]

df_gold_standard_full

,text,polarity
0,No. I still see the wrong twins. * https://gi...,neutral
1,Reverted.,neutral
2,You can leave a queue while in queue ? (before...,neutral
3,Didn't look at SpellTargetRestrictions XD,positive
4,Not sure about what kind of line lengths the p...,neutral
...,...,...
7117,"Yeah, I'm capable of working around it. A set...",negative
7118,"Looks fine to me! On Jul 8, 2013, at 10:12 PM...",positive
7119,Formula updated to address this (would be nice...,positive
7120,I trust you,positive


### Test/Train split
Note that, in the paper, the GitHub Gold Standard dataset is split into a 30% test set and a 70% train set, on which DEVA (among other models) is run. To keep this replication as close as possible to the original, we also perform this partition with `scikit` and then use the test set for evaluation. Note that the paper does not mention a specific seed used for the split, so we use our own seed and mention that the results won't be 100% the same as the paper (however, the macro-averages should match closely).

In [5]:
_, df_gold_standard = train_test_split(df_gold_standard_full, test_size=0.3, random_state=42)
df_gold_standard = df_gold_standard.reset_index(drop=True)
df_gold_standard

,text,polarity
0,Guess there is a typo here: `translate`.,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral
2,Thank you Vlad. Your contribution to Mangos an...,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative
4,"Ok, so let's be paranoid.",negative
...,...,...
2132,moved into encoders/decoders,neutral
2133,"I understand. In that case, we would need to h...",negative
2134,that is a good idea.,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral


## Coutinho et al., 2024
JSON format, so some normalization is required.

In [6]:
text = COUTINHO.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_premo = pd.json_normalize(data, sep="_")

df_premo['polarity'] = df_premo['part2_aggregate_polarity'].where(
    df_premo['part2_aggregate_polarity'] != 'undefined',
    other=df_premo['discussion_polarity']
)
df_premo = df_premo[df_premo['polarity'].notna() & (df_premo['polarity'] != 'undefined')]

# normalize to same structure as gold standard dataset
df_premo = df_premo[['clean_message', 'polarity']].rename(columns={'clean_message': 'text'})

df_premo

,text,polarity
0,did you get a chance to follow up on the issue...,neutral
1,Thank you for this PR. This is an interesting...,neutral
2,", oh sorry for this, my fault, i'll be more c...",negative
3,I've added so that we can add documentation r...,neutral
4,Applications can still use or qualifiers to g...,negative
...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative
1787,"<!-- First of all, thank you for your contribu...",neutral
1788,"there was a typo in the comment, also made a s...",neutral
1789,Fixed some typos. Added the ghost button type....,neutral


## Romanian Gold Standard
The dataset collected and introduced in this study. JSON format, so normalization is required.

In [7]:
text = RO_GOLD.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_ro_gold = pd.json_normalize(data, sep="_")

# normalize to same structure as gold standard dataset
df_ro_gold = df_ro_gold[['text', 'polarity']]

df_ro_gold

,text,polarity
0,daca buseste proiectul va fi vina ta,negative
1,OK. Algoritmul este implementat corect după in...,neutral
2,Arata foarte bine 👍,positive
3,bravo buna treaba,positive
4,"Looking good, de vazut doar faza cu acolada de...",neutral
...,...,...
1947,aici nu trebuie sa folosesti handleChange ci s...,neutral
1948,nu era mai simplu: sa imparti cu 20LL din prim...,neutral
1949,Da asa e. Am făcut-o ca s ama prind cum merge ...,neutral
1950,"Avea un font diferit,asa am crezut ca trebuie ...",neutral


### Label distribution

In [8]:
pol_c = Counter(r['polarity'] for r in data)
total = len(data)
print(f'final polarity distribution ({total} entries):')
for lab in ['positive', 'neutral', 'negative']:
    n = pol_c.get(lab, 0)
    print(f'  {lab:10s} {n:5d} ({n/total:.2%})')
unresolved = pol_c.get(None, 0)
if unresolved:
    print(f'  unresolved {unresolved:5d} ({unresolved/total:.2%})')

final polarity distribution (1952 entries):
  positive     135 (6.92%)
  neutral     1612 (82.58%)
  negative     205 (10.50%)


In [9]:
emo_c = Counter(a['primary_emotion'] for r in data for a in r['annotators'] if a['primary_emotion'])
total_emo = sum(emo_c.values())
print(f'primary emotion distribution (rater-level, {total_emo} votes):')
for emo, n in emo_c.most_common():
    print(f'  {emo:10s} {n:5d} ({n/total_emo:.2%})')

primary emotion distribution (rater-level, 5828 votes):
  none        4604 (79.00%)
  joy          390 (6.69%)
  anger        329 (5.65%)
  fear         256 (4.39%)
  sadness      188 (3.23%)
  love          61 (1.05%)


### Inter-annotator agreement (Fleiss kappa)

In [10]:
CATS = ['positive', 'neutral', 'negative']
ORDER = {'positive': 0, 'neutral': 1, 'negative': 2}

def build_matrix(records):
    rows = []
    for r in records:
        pols = [a['polarity'] for a in r['annotators'] if a['polarity'] is not None]
        if len(pols) != 3:
            continue
        rows.append([pols.count(c) for c in CATS])
    return np.array(rows, dtype=float)

def fleiss_kappa(M):
    N, k = M.shape
    n = M.sum(axis=1)[0]
    P_i = ((M ** 2).sum(axis=1) - n) / (n * (n - 1))
    P_bar = P_i.mean()
    p_j = M.sum(axis=0) / (N * n)
    P_e = (p_j ** 2).sum()
    return (P_bar - P_e) / (1 - P_e), P_bar, P_e

def fleiss_kappa_weighted(M, W):
    N, k = M.shape
    n = M.sum(axis=1)[0]
    P_i = np.zeros(N)
    for idx in range(N):
        row = M[idx]
        s = 0.0
        for a in range(k):
            for b in range(k):
                s += W[a, b] * (row[a] * (row[a] - 1) if a == b else row[a] * row[b])
        P_i[idx] = s / (n * (n - 1))
    P_bar = P_i.mean()
    p_j = M.sum(axis=0) / (N * n)
    P_e = sum(W[a, b] * p_j[a] * p_j[b] for a in range(k) for b in range(k))
    return (P_bar - P_e) / (1 - P_e)

M = build_matrix(data)
print(f'items with full 3-rater polarity coverage: {M.shape[0]}')

k_un, Pbar, Pe = fleiss_kappa(M)
print(f'fleiss kappa (unweighted):         {k_un:.4f}   [P_bar={Pbar:.4f}, P_e={Pe:.4f}]')

W_lin = np.array([[1 - abs(ORDER[a] - ORDER[b]) / 2 for b in CATS] for a in CATS])
W_quad = np.array([[1 - (ORDER[a] - ORDER[b]) ** 2 / 4 for b in CATS] for a in CATS])
print(f'fleiss kappa (linear-weighted):    {fleiss_kappa_weighted(M, W_lin):.4f}')
print(f'fleiss kappa (quadratic-weighted): {fleiss_kappa_weighted(M, W_quad):.4f}')

items with full 3-rater polarity coverage: 1924
fleiss kappa (unweighted):         0.4578   [P_bar=0.8091, P_e=0.6479]
fleiss kappa (linear-weighted):    0.4756
fleiss kappa (quadratic-weighted): 0.5060


### Polarity by comment type

In [11]:
# rebuild dataframe retaining comment_type (it is dropped from df_ro_gold above)
with open(RO_GOLD, 'r', encoding='utf-8') as f:
    _ro_raw = json.load(f)

df_ro_typed = pd.DataFrame([
    {'comment_type': r['comment_type'], 'polarity': r['polarity']}
    for r in _ro_raw
])

POL_ORDER = ['positive', 'neutral', 'negative']
TYPE_ORDER = ['review_comment', 'issue_comment', 'review']

# raw counts contingency table
ct_counts = pd.crosstab(df_ro_typed['comment_type'], df_ro_typed['polarity'])
ct_counts = ct_counts.reindex(index=TYPE_ORDER, columns=POL_ORDER)
ct_counts['total'] = ct_counts.sum(axis=1)
ct_counts.loc['overall'] = ct_counts.sum(axis=0)

# row-normalised shares (per-type polarity distribution)
ct_shares = pd.crosstab(df_ro_typed['comment_type'], df_ro_typed['polarity'], normalize='index')
ct_shares = ct_shares.reindex(index=TYPE_ORDER, columns=POL_ORDER)
ct_shares.loc['overall'] = df_ro_typed['polarity'].value_counts(normalize=True).reindex(POL_ORDER)

# combined overview
N_total = int(ct_counts.loc['overall', 'total'])
summary = pd.DataFrame({
    'n':        ct_counts['total'].astype(int),
    'share':    (ct_counts['total'] / N_total).map('{:.1%}'.format),
    'positive': ct_shares['positive'].map('{:.1%}'.format),
    'neutral':  ct_shares['neutral'].map('{:.1%}'.format),
    'negative': ct_shares['negative'].map('{:.1%}'.format),
})
summary

,n,share,positive,neutral,negative
comment_type,,,,,
review_comment,1375,70.4%,2.6%,88.5%,8.9%
issue_comment,384,19.7%,11.7%,74.5%,13.8%
review,193,9.9%,28.0%,56.5%,15.5%
overall,1952,100.0%,6.9%,82.6%,10.5%


### Chi-square test of independence

To check whether the polarity distribution differs significantly across comment types (and not just from sampling noise), we run Pearson's chi-square test of independence on the 3x3 contingency table above.

- $H_0$ (null): comment type and polarity are independent. The polarity distribution is the same across types.
- $H_1$ (alternative): at least one type has a different polarity distribution.

Chi-square's standard validity rule requires every expected cell count to be at least 5 (or more leniently: no cell below 1 and at least 80% of cells at or above 5). If those assumptions are violated, the fallback is Fisher's exact test (Freeman-Halton extension for I x J tables). We also report Cramer's V as an effect-size measure: 0 means independence, 1 means perfect association.

In [12]:
# the 3x3 observed counts (drop the totals row/column added above)
observed = ct_counts.drop(index='overall', columns='total')

chi2, pval, dof, expected = stats.chi2_contingency(observed.values)
expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)

print('Observed counts (3x3):')
print(observed)
print('\nExpected counts under H_0:')
print(expected_df.round(1))

min_expected = float(expected.min())
n_cells_lt_5 = int((expected < 5).sum())
print(f'\nmin expected count: {min_expected:.2f}')
print(f'cells with expected < 5: {n_cells_lt_5} of {expected.size}')

n_obs = int(observed.values.sum())
cramers_v = (chi2 / (n_obs * (min(observed.shape) - 1))) ** 0.5

print(f'\nchi^2 = {chi2:.3f}')
print(f'dof   = {dof}')
print(f'p     = {pval:.3e}')
print(f"Cramer's V = {cramers_v:.3f}")

if min_expected >= 5:
    print('\nChi-square assumptions satisfied (all expected counts >= 5). No fallback needed.')
else:
    print('\nWARNING: chi-square assumptions violated. Fall back to Fisher s exact (Freeman-Halton for I x J tables).')

Observed counts (3x3):
polarity        positive  neutral  negative
comment_type                               
review_comment        36     1217       122
issue_comment         45      286        53
review                54      109        30

Expected counts under H_0:
polarity        positive  neutral  negative
comment_type                               
review_comment      95.1   1135.5     144.4
issue_comment       26.6    317.1      40.3
review              13.3    159.4      20.3

min expected count: 13.35
cells with expected < 5: 0 of 9

chi^2 = 210.299
dof   = 4
p     = 2.291e-44
Cramer's V = 0.232

Chi-square assumptions satisfied (all expected counts >= 5). No fallback needed.


In [13]:
# full disagreements: entries where the 3 raters each picked a different polarity (1-1-1 split)
disagreement_rows = []
for r in _ro_raw:
    pols = [a.get('polarity') for a in r.get('annotators', []) if a.get('polarity')]
    if len(pols) == 3 and len(set(pols)) == 3:
        disagreement_rows.append({
            'comment_type': r.get('comment_type'),
            'text': r.get('text'),
            'rater_polarities': ', '.join(sorted(pols)),
            'final_polarity': r.get('polarity'),
        })

df_full_disagreement = pd.DataFrame(disagreement_rows)
print(f'Full polarity disagreements (every rater picked a different class): {len(df_full_disagreement)} of {len(_ro_raw)} entries ({len(df_full_disagreement)/len(_ro_raw):.1%})')
df_full_disagreement

Full polarity disagreements (every rater picked a different class): 12 of 1952 entries (0.6%)


,comment_type,text,rater_polarities,final_polarity
0,review,Da cam destept domnu bot,"negative, neutral, positive",neutral
1,issue_comment,aaa .. si numele repo-ului tre sa fie numele p...,"negative, neutral, positive",neutral
2,issue_comment,"arata bine pana acuma, muta si interfatele de ...","negative, neutral, positive",neutral
3,issue_comment,"Nu mai modific, is mult prea rare cazurile in ...","negative, neutral, positive",negative
4,issue_comment,"@AdrianCostescu, sa rezolvi te rog conflictele...","negative, neutral, positive",neutral
5,issue_comment,ba trebuia zoom-in cu Space si out cu Shift \...,"negative, neutral, positive",negative
6,issue_comment,"@ionel-popescu93, înțeleg ce ai întrebat tu, d...","negative, neutral, positive",negative
7,review_comment,"E facut ciudat, dar functioneaza si nici nu ma...","negative, neutral, positive",positive
8,review_comment,De fapt tu nici nu folosești componenta pe car...,"negative, neutral, positive",neutral
9,review_comment,"posibil sa ai dreptate, pe mine deja ma baga i...","negative, neutral, positive",negative


# DEVA Experiments (English datasets)

DEVA expects an input .txt file with the message to be analyzed and its ID. To demonstrate, this is the example input they provide with the tool:
```
58204 Yes, that's correct.
32957 Challenge accepted! :)
45982 No problem.  It was my pleasure, and thanks for all the help.
63699 Still need tests  Neutral
14350 My mistake, sorry
```

It outputs the original input, as well as three extra columns: its valence score, its arousal score, and an emotional state label derived from the combination of both.

For the purposes of this replication study, however, the benchmark reduces sentiment to a three-class polarity task (positive, negative, neutral), so only the valence score is used. This maps directly to Novielli et al. replication's $±[1,3]$ schema:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

The output for the example input looks as follows:
```
58204 Yes, that's correct.  -2  -3    DEPRESSED
32957 Challenge accepted! :)  2   5   EXCITED
45982 No problem.  It was my pleasure, and thanks for all the help.   4   3   EXCITED
63699 Still need tests  Neutral   1   -2    NEUTRAL (LA)
14350 My mistake, sorry   -2  1   NEG VALENCE
```

Therefore, to run DEVA on the datasets collected above, we need to generate valid input, and then parse the output into dataframes.

In [32]:
def write_deva_input(df, output_path, text_col='text'):
    """Write a DEVA-compatible input .txt file from a dataframe."""
    df_out = df[[text_col]].copy()
    df_out.to_csv(output_path, sep='\t', header=False, index=True)

def parse_deva_output(filepath):
    valences, arousals, labels = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:                                                                         
        for line in f:                                                                                                       
            parts = line.rstrip('\n').rsplit('\t', 3)                                                                        
            if len(parts) == 4:                                                                                              
                _, valence, arousal, label = parts
                valences.append(int(valence.strip()))                                                                        
                arousals.append(int(arousal.strip()))
                labels.append(label.strip())                                                                                 
    return valences, arousals, labels

In [33]:
write_deva_input(df_gold_standard, DEVA_INPUT_DIR / "gold_standard.txt")
write_deva_input(df_premo, DEVA_INPUT_DIR / "premo.txt")

In [34]:
v, a, l = parse_deva_output(DEVA_OUTPUT_DIR / "gold_standard_out.txt")
df_gold_standard['valence'] = v

df_gold_standard.head(5)

,text,polarity,valence
0,Guess there is a typo here: `translate`.,neutral,1
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,4
2,Thank you Vlad. Your contribution to Mangos an...,positive,3
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,-3
4,"Ok, so let's be paranoid.",negative,1


In [35]:
v, a, l = parse_deva_output(DEVA_OUTPUT_DIR / "premo_out.txt")
df_premo['valence'] = v

df_premo.head(5)

,text,polarity,valence
0,did you get a chance to follow up on the issue...,neutral,1
1,Thank you for this PR. This is an interesting...,neutral,-3
2,", oh sorry for this, my fault, i'll be more c...",negative,-2
3,I've added so that we can add documentation r...,neutral,1
4,Applications can still use or qualifiers to g...,negative,-2


## Mapping DEVA results to polarity labels
As mentioned above, the following mapping is used:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

In [36]:
def valence_to_polarity(v):
    if v > 1:
        return 'positive'
    elif v < -1:
        return 'negative'
    else:
        return 'neutral'

In [37]:
df_gold_standard['deva_polarity'] = df_gold_standard['valence'].apply(valence_to_polarity)
df_gold_standard = df_gold_standard.drop(columns=['valence'], errors='ignore')
df_gold_standard

,text,polarity,deva_polarity
0,Guess there is a typo here: `translate`.,neutral,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,positive
2,Thank you Vlad. Your contribution to Mangos an...,positive,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,negative
4,"Ok, so let's be paranoid.",negative,neutral
...,...,...,...
2132,moved into encoders/decoders,neutral,neutral
2133,"I understand. In that case, we would need to h...",negative,negative
2134,that is a good idea.,positive,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral,neutral


In [38]:
df_premo['deva_polarity'] = df_premo['valence'].apply(valence_to_polarity)
df_premo = df_premo.drop(columns=['valence'], errors='ignore')
df_premo

,text,polarity,deva_polarity
0,did you get a chance to follow up on the issue...,neutral,neutral
1,Thank you for this PR. This is an interesting...,neutral,negative
2,", oh sorry for this, my fault, i'll be more c...",negative,negative
3,I've added so that we can add documentation r...,neutral,neutral
4,Applications can still use or qualifiers to g...,negative,negative
...,...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative,neutral
1787,"<!-- First of all, thank you for your contribu...",neutral,negative
1788,"there was a typo in the comment, also made a s...",neutral,neutral
1789,Fixed some typos. Added the ghost button type....,neutral,negative


## Calculating precision, recall, F1 for DEVA results

For each class $cls$ of $classes=[positive, negative, neutral]$, we compute:
$$\text{precision}(cls) = \frac{TP}{TP + FP}\;\;\;\;\;\;\;\;\text{(of all messages predicted as class cls, the fraction that are truly cls)}$$
$$\text{recall}(cls) = \frac{TP}{TP + FN}\;\;\;\;\;\;\;\;\text{(of all messages that are truly of class cls, the fraction correctly identified)}$$
$$\text{f1}(cls) = 2\times\frac{\text{precision}(cls) \times \text{recall}(cls)}{\text{precision}(cls) + \text{recall}(cls)}\;\;\;\;\;\;\;\;\text{(harmonic mean of precision and recall)},$$
where for each class $cls$, TP is the number of messages correctly predicted as $cls$, FP is the number of messages incorrectly predicted as $cls$, and FN is the number of messages that belong to $cls$ but were predicted as a different class.

Moreover, for each metric, we compute macro averages and weighted averages:
$$\text{precision}_{\text{macro}} = \frac{1}{|classses|}\sum_{cls}\text{precision}(cls)$$
$$\text{precision}_{\text{weighted}} = \frac{\sum_{cls}\text{precision}(cls)\times|cls|}{\sum_{cls}|cls|}$$

We also aggregate TP, FP, FN across all classes to calculate an overall accuracy: micro-average. We use the following formulae:

$$\text{precision}_\text{micro}=\frac{\sum_{cls}TP_{cls}}{\sum_{cls}(TP_{cls}+FP_{cls}}$$
$$\text{recall}_\text{micro}=\frac{\sum_{cls}TP_{cls}}{\sum_{cls}(TP_{cls}+FN_{cls})}$$
$$\text{f1}_\text{micro}=2\times\frac{\text{precision}_\text{micro}\times \text{recall}_\text{micro}}{\text{precision}_\text{micro}+ \text{recall}_\text{micro}}$$

Note that in a multiclass setting where every sample gets exactly one label (as is in our case), $\sum TP_{cls} = \sum(TP_{cls} + FP_{cls}) = \sum(TP_{cls} + FN_{cls})$ (total number of samples). Therefore, $\text{precision}_\text{micro} = \text{recall}_\text{micro} = \text{f1}_\text{micro}$, which serves as an overall accuracy.

In [39]:
def compute_metrics(df, dataset_name, pred_col='deva_polarity', gold_col='polarity'):
    classes = ['positive', 'negative', 'neutral']
    rows = []

    for cls in classes:
        tp = ((df[pred_col] == cls) & (df[gold_col] == cls)).sum()
        fp = ((df[pred_col] == cls) & (df[gold_col] != cls)).sum()
        fn = ((df[pred_col] != cls) & (df[gold_col] == cls)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        rows.append({
            'dataset': dataset_name,
            'class': cls,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': (df[gold_col] == cls).sum()
        })


    # macro average (simple mean across classes, like Coutinho et al.)
    macro = pd.DataFrame(rows)
    rows.append({
        'dataset': dataset_name,
        'class': 'macro avg',
        'precision': macro['precision'].mean(),
        'recall': macro['recall'].mean(),
        'f1': macro['f1'].mean(),
        'support': len(df)
    })

    # weighted average (mean weighted by class support)
    total = len(df)
    rows.append({
        'dataset': dataset_name,
        'class': 'weighted avg',
        'precision': (macro['precision'] * macro['support']).sum() / total,
        'recall':    (macro['recall']    * macro['support']).sum() / total,
        'f1':        (macro['f1']        * macro['support']).sum() / total,
        'support': total
    })

    # micro average (aggregate TP/FP/FN across all classes first)
    tp_sum = sum(((df[pred_col] == c) & (df[gold_col] == c)).sum() for c in classes)
    fp_sum = sum(((df[pred_col] == c) & (df[gold_col] != c)).sum() for c in classes)
    fn_sum = sum(((df[pred_col] != c) & (df[gold_col] == c)).sum() for c in classes)
    micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) > 0 else 0
    micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) > 0 else 0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    rows.append({
        'dataset': dataset_name,
        'class': 'micro avg',
        'precision': micro_p,
        'recall': micro_r,
        'f1': micro_f,
        'support': total
    })


    return pd.DataFrame(rows)

In [40]:
metrics_gold_standard = compute_metrics(df_gold_standard, 'gold_standard')
metrics_premo = compute_metrics(df_premo, 'premo')

metrics_all = pd.concat([metrics_gold_standard, metrics_premo], ignore_index=True)
metrics_all

,dataset,class,precision,recall,f1,support
0,gold_standard,positive,0.712733,0.745130,0.728571,616
1,gold_standard,negative,0.663235,0.718153,0.689602,628
2,gold_standard,neutral,0.801968,0.730123,0.764361,893
3,gold_standard,macro avg,0.725979,0.731135,0.727512,2137
4,gold_standard,weighted avg,0.735476,0.730931,0.732075,2137
5,gold_standard,micro avg,0.730931,0.730931,0.730931,2137
6,premo,positive,0.664903,0.723608,0.693015,521
7,premo,negative,0.484076,0.703704,0.573585,432
8,premo,neutral,0.798658,0.568019,0.663877,838
9,premo,macro avg,0.649212,0.665110,0.643492,1791


Note: the weighted average in the results is higher than the macro average mostly because the model seems to do much better on neutral text.

Note: the numbers are slightly different for the gs_gh set from the paper by Novielli et al. This is because I believe they run the experiemnts on the test set only (30% of the whole dataset), while I run it on the whole dataset. Moreover, the macro- and micro- averages are the same between both sets of results, confirming the validity of my replication.

Some takeaways:
- for the Github Golden Standard dataset, DEVA does relatively well, with a macro-average F1 score of 0.725. Neutral is the easiest class to predict (F1 score of 0.766), which makes sense given lexicon-based tools tend to default toward neutral when no strong sentiment words are found
- for the Coutinho dataset, DEVA struggles more: a macro-average F1 score of just 0.643. The difference is especially striking in the negative class performance, as the precision is very low (0.484). This means that DEVA is over-predicting negative sentiment on this dataset, flagging many non-negative items as negative. Interestingly, Coutinho et al. state that _"unlike positive messages, negative messages were frequently very subtle, sometimes even employing irony or sarcasm, which has been noted by previous works as a problem for these tools"_. However, this is not really the case? As based on the results, it seems like DEVA is very trigger-happy with negative labelling, possibly due to technical vocabulary. **This is something worth analyzing perhaps.**
- on the GH GS dataset, DEVA is conservative with neutral labels: it has high precision but moderate recall. However, on the coutinho dataset, it has high precision but low recall -- meaning it's missing a lot of neutral cases, likely identifying them as negative. In both cases, DEVA under-predicts neutral, but it is quite precise when it does assign neutral.

**Why does DEVA seem to be so precise with neutral labels, but also so conservative?**
It's mostly down to how DEVA works under the hood. DEVA computes the overall valence score of a text as whichever of the max positive score or max negative score has the larger absolute value. So, for a text to be classified as neutral, every sentiment-bearing word must score +- 1, or no words match the dictionary at all. Any word scoring +2 or -2 anywhere in the text will pull the whole document to positive or negative, regardless of the neutral context.

# LLM Experiments
We use three models: Gemma3:12b, Phi4:14b, and Llama3.1:8b.

In [41]:
# config

MODELS = ["gemma3:12b", "phi4:14b", "llama3.1:8b"]
CACHE_DIR = ROOT_DIR / "cache" / "llm"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONSECUTIVE_ERRORS = 3

In [42]:
# cache helpers

def cache_path(model: str, prompt_id: int, dataset: str) -> Path:
    safe_name = model.replace(":", "_").replace("/", "_")
    return CACHE_DIR / f"{safe_name}_prompt_{prompt_id}_{dataset}.json"

def load_cache(model: str, prompt_id: int, dataset: str) -> dict:
    path = cache_path(model, prompt_id, dataset)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            entries = json.load(f)
        return {e["text"]: (e.get("raw_message", ""), e["label"]) for e in entries}
    return {}

def save_cache(model: str, prompt_id: int, dataset: str, results: dict):
    entries = [
        {"text": text, "raw_message": raw, "label": label}
        for text, (raw, label) in results.items()
    ]
    with open(cache_path(model, prompt_id, dataset), "w", encoding="utf-8") as f:
        json.dump(entries, f, ensure_ascii=False, indent=1)

## Label Parser
Prompt 0 has the explicit "Return label only" instruction at the end, which makes label parsing very straight forward. However, prompts 1 and 2 lack this instruction (after work by Zhang et al.), which makes it more difficult to capture the actual label the LLM outputted. This is especially true for the (translated) Romanian prompts 1 and 2, where the models tend to be more verbose and less direct when giving their final label. Therefore, we use a single, robust parser. It scans the entire response rather than only the final line, and runs four structured passes before falling back to token-level matching:

- Explicit `Etichetă:` / `Label:` / `Sentiment:` marker, last match wins
- Markdown bold declaration like `**Negativ**` or `**(2) neutral**`, first match wins
- Numeric prefix declarations like `(2) neutral`, last match wins
- Bare numeric option at the start of the response like `(3)` on its own, mapped to the prompt convention (1)=positive, (2)=neutral, (3)=negative
- Token scan as final fallback

The parser recognises Romanian label words (`pozitiv`, `neutru`/`neutră`, `negativ`) in addition to the English ones, as well as the anglicised `positiv` that one of the models emits in place of `pozitiv`. If nothing matches anywhere, the response is conservatively labelled `neutral` and recorded as `default` so it can be flagged in downstream analysis.


In [43]:
LABEL_PATTERN = re.compile(
    r'(?:etichet[aă]|label|sentiment)\s*[:\-]\s*\**\s*(?:\(\s*\d+\s*\)\s*)?\**\s*([a-zà-ɏ]+)',
    re.IGNORECASE,
)
BOLD_LABEL_PATTERN = re.compile(
    r'\*\*\s*(?:\(\s*\d+\s*\)\s*)?'
    r'(positive|negative|neutral|pozitiv\w*|positiv\w*|negativ\w*|neutr[uă]\w*)'
    r'\s*\*\*',
    re.IGNORECASE,
)
NUMBERED_LABEL_PATTERN = re.compile(
    r'\(\s*\d+\s*\)\s*'
    r'(positive|negative|neutral|pozitiv\w*|positiv\w*|negativ\w*|neutr[uă]\w*)',
    re.IGNORECASE,
)
BARE_NUMBERED_PATTERN = re.compile(r'^\s*\(\s*([123])\s*\)\s*(?:[-.]|$)')
_BARE_NUM_TO_LABEL = {'1': 'positive', '2': 'neutral', '3': 'negative'}

def _map_token(token):
    t = token.lower().strip('.,:;!?"\'`*()[]{}')
    if 'pozitiv' in t or 'positiv' in t:
        return 'positive'
    if 'neutru' in t or 'neutră' in t:
        return 'neutral'
    if 'negativ' in t:
        return 'negative'
    if 'positive' in t:
        return 'positive'
    if 'neutral' in t:
        return 'neutral'
    return None

def extract_label_with_source(response):
    text = response.strip().lower()
    if not text:
        return 'neutral', 'default'

    for m in reversed(list(LABEL_PATTERN.finditer(text))):
        mapped = _map_token(m.group(1))
        if mapped:
            return mapped, 'marker'

    for m in BOLD_LABEL_PATTERN.finditer(text):
        mapped = _map_token(m.group(1))
        if mapped:
            return mapped, 'bold'

    for m in reversed(list(NUMBERED_LABEL_PATTERN.finditer(text))):
        mapped = _map_token(m.group(1))
        if mapped:
            return mapped, 'numbered'

    m = BARE_NUMBERED_PATTERN.match(text)
    if m:
        return _BARE_NUM_TO_LABEL[m.group(1)], 'bare_numbered'

    for line in text.splitlines():
        for token in line.split():
            mapped = _map_token(token)
            if mapped:
                return mapped, 'token'

    return 'neutral', 'default'

def extract_label(response):
    return extract_label_with_source(response)[0]


## Prompting and Classification

In [44]:
def classify(text: str, model: str, task_prompt: str, template: dict) -> tuple[str, str]:
    resp = ollama.chat(
        model=model,
        messages=[
            {"role": "system",    "content": template["system"]},
            {"role": "user",      "content": task_prompt.strip()},
            {"role": "assistant", "content": template["ack"]},
            {"role": "user",      "content": template["user_template"].format(text=text)},
        ],
        options={"temperature": 0},
        think=False,
    )
    raw = resp.message.content
    return raw, template["extract"](raw)

def classify_with_cache(texts, model: str, prompt_id, dataset: str, task_prompt: str, template: dict) -> list[str]:
    cached = load_cache(model, prompt_id, dataset)
    results = dict(cached)  # copy
    labels = []
    consecutive_errors = 0

    for text in tqdm(texts, desc=f"{model} (prompt {prompt_id})"):
        if text in results:
            labels.append(results[text][1])
            consecutive_errors = 0
            continue

        try:
            raw, label = classify(text, model, task_prompt, template)
            results[text] = (raw, label)
            labels.append(label)
            consecutive_errors = 0
        except Exception as e:
            consecutive_errors += 1
            print(f"\n  Error on {model}: {e}")
            results[text] = ('', 'error')
            labels.append("error")

            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print(f"  {MAX_CONSECUTIVE_ERRORS} consecutive errors: skipping {model}")
                labels.extend(["error"] * (len(texts) - len(labels)))
                break

        # save periodically (every 25 items)
        if len(labels) % 25 == 0:
            save_cache(model, prompt_id, dataset, results)

    save_cache(model, prompt_id, dataset, results)
    return labels

def run_classification(df, model: str, prompt_id, dataset: str, task_prompt: str, template: dict):
    col = f"{model.replace(':', '_').replace('/', '_')}_prompt_{prompt_id}"
    df[col] = classify_with_cache(df["text"].tolist(), model, prompt_id, dataset, task_prompt, template)

For this analysis, we will be using the prompts of Zhang et al.

**Prompt 0:**

Please perform Sentiment Classification task. Given the sentence from GitHub pull request code review comments, assign a sentiment label from ['negative', 'neutral', 'positive']. Return label only without any other text.

**Prompt 1**

Please categorize the sentiment expressed in the following sentence from GitHub pull request code review comments as either (1) positive, (2) neutral, or (3) negative.

**Prompt 2**

I will give you a sentence from GitHub pull request code review comments. You need to reply with the sentiment expressed in the following sentence, either (1) positive, (2) neutral, or (3) negative.

In [45]:
PROMPT_0 ="""
Please perform Sentiment Classification task. Given the sentence from GitHub pull request code review comments, assign a sentiment label from ['negative', 'neutral', 'positive']. Return label only without any other text.
"""

PROMPT_1 ="""
Please categorize the sentiment expressed in the following sentence from GitHub pull request code review comments as either (1) positive, (2) neutral, or (3) negative.
"""

PROMPT_2 ="""
I will give you a sentence from GitHub pull request code review comments. You need to reply with the sentiment expressed in the following sentence, either (1) positive, (2) neutral, or (3) negative.
"""

PROMPTS = {0: PROMPT_0, 1: PROMPT_1, 2: PROMPT_2}

Furthermore, for the Romanian dataset, we also translate the prompts of Zhang et al. into Romanian. We do this to observe whether (and how much) this affects results.

**Prompt 0 (translated):**

Efectuează sarcina de clasificare a sentimentului. Pentru propoziția dată din comentarii de code review ale unui pull request de pe GitHub, atribuie o etichetă de sentiment din ['negative', 'neutral', 'positive']. Returnează doar eticheta, fără alt text.

**Prompt 1 (translated):**

Clasifică sentimentul exprimat în următoarea propoziție din comentarii de code review ale unui pull request de pe GitHub drept (1) positive, (2) neutral sau (3) negative.

**Prompt 2 (translated):**

Îți voi da o propoziție din comentarii de code review ale unui pull request de pe GitHub. Trebuie să răspunzi cu sentimentul exprimat în propoziția următoare, fie (1) positive, (2) neutral, fie (3) negative.

In [46]:
PROMPT_0_RO="""
Efectuează sarcina de clasificare a sentimentului. Pentru propoziția dată din comentarii de code review ale unui pull request de pe GitHub, atribuie o etichetă de sentiment din ['negative', 'neutral', 'positive']. Returnează doar eticheta, fără alt text.
"""

PROMPT_1_RO="""
Clasifică sentimentul exprimat în următoarea propoziție din comentarii de code review ale unui pull request de pe GitHub drept (1) positive, (2) neutral sau (3) negative.
"""

PROMPT_2_RO="""
Îți voi da o propoziție din comentarii de code review ale unui pull request de pe GitHub. Trebuie să răspunzi cu sentimentul exprimat în propoziția următoare, fie (1) positive, (2) neutral, fie (3) negative.
"""

PROMPTS_RO = {0: PROMPT_0_RO, 1: PROMPT_1_RO, 2: PROMPT_2_RO}

In [47]:
SYSTEM_PROMPT = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)

SYSTEM_PROMPT_RO = (
    "O conversație între un utilizator curios și un asistent de inteligență artificială. "
    "Asistentul oferă răspunsuri utile, detaliate și politicoase la întrebările utilizatorului."
)

TEMPLATE_EN = {
    "system": SYSTEM_PROMPT,
    "ack": "Sure!",
    "user_template": "Sentence: {text}\nLabel:",
    "extract": extract_label,
}

TEMPLATE_RO = {
    "system": SYSTEM_PROMPT_RO,
    "ack": "Sigur!",
    "user_template": "Propoziție: {text}\nEtichetă:",
    "extract": extract_label,
}


## Running LLMs

In [48]:
# work on copies so the original dataframes stay clean
df_gold_standard_llm = df_gold_standard[['text', 'polarity']].copy()
df_premo_llm = df_premo[['text', 'polarity']].copy()
df_ro_gold_llm = df_ro_gold[['text', 'polarity']].copy()

In [49]:
# Gold standard
for model in MODELS:
    for prompt_id, prompt in PROMPTS.items():
        run_classification(
            df_gold_standard_llm,
            model,
            prompt_id,
            "gold_standard",
            prompt,
            TEMPLATE_EN,
        )

llama3.1:8b (prompt 2): 100%|██████████| 2137/2137 [00:00<00:00, 521057.30it/s]


In [50]:
# PRemo
for model in MODELS:
    for prompt_id, prompt in PROMPTS.items():
        run_classification(
            df_premo_llm,
            model,
            prompt_id,
            "premo",
            prompt,
            TEMPLATE_EN,
        )

llama3.1:8b (prompt 2): 100%|██████████| 1791/1791 [00:00<00:00, 552068.68it/s]


In [51]:
# Romanian gold standard: run with both English and Romanian prompts/templates
for model in MODELS:
    for prompt_id, prompt in PROMPTS.items():
        run_classification(
            df_ro_gold_llm,
            model,
            prompt_id,
            "ro_gold_standard",
            prompt,
            TEMPLATE_EN,
        )
    for prompt_id, prompt in PROMPTS_RO.items():
        run_classification(
            df_ro_gold_llm,
            model,
            f"{prompt_id}_ro",
            "ro_gold_standard",
            prompt,
            TEMPLATE_RO,
        )

llama3.1:8b (prompt 2_ro): 100%|██████████| 1952/1952 [00:00<00:00, 489377.25it/s]


In [52]:
df_gold_standard_llm

,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2
0,Guess there is a typo here: `translate`.,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,negative,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,neutral,neutral,neutral,neutral,negative,negative,negative,neutral,negative
2,Thank you Vlad. Your contribution to Mangos an...,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,neutral,neutral,negative,neutral,negative,negative,negative,negative,negative
4,"Ok, so let's be paranoid.",negative,negative,negative,negative,neutral,neutral,neutral,negative,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...
2132,moved into encoders/decoders,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral
2133,"I understand. In that case, we would need to h...",negative,negative,neutral,negative,neutral,negative,negative,negative,negative,negative
2134,that is a good idea.,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral,neutral,neutral,neutral,negative,negative,negative,negative,negative,negative


In [53]:
df_premo_llm

,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2
0,did you get a chance to follow up on the issue...,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,positive
1,Thank you for this PR. This is an interesting...,neutral,neutral,neutral,neutral,negative,negative,negative,negative,negative,negative
2,", oh sorry for this, my fault, i'll be more c...",negative,negative,negative,negative,neutral,negative,negative,negative,negative,negative
3,I've added so that we can add documentation r...,neutral,positive,positive,positive,positive,positive,positive,neutral,positive,neutral
4,Applications can still use or qualifiers to g...,negative,negative,neutral,negative,negative,negative,negative,negative,negative,negative
...,...,...,...,...,...,...,...,...,...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral
1787,"<!-- First of all, thank you for your contribu...",neutral,positive,positive,positive,positive,positive,positive,positive,positive,positive
1788,"there was a typo in the comment, also made a s...",neutral,negative,negative,neutral,neutral,neutral,neutral,negative,negative,negative
1789,Fixed some typos. Added the ghost button type....,neutral,neutral,neutral,neutral,neutral,neutral,neutral,negative,neutral,neutral


In [54]:
df_ro_gold_llm

,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,gemma3_12b_prompt_0_ro,gemma3_12b_prompt_1_ro,gemma3_12b_prompt_2_ro,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,phi4_14b_prompt_0_ro,phi4_14b_prompt_1_ro,phi4_14b_prompt_2_ro,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2,llama3.1_8b_prompt_0_ro,llama3.1_8b_prompt_1_ro,llama3.1_8b_prompt_2_ro
0,daca buseste proiectul va fi vina ta,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative,negative
1,OK. Algoritmul este implementat corect după in...,neutral,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,neutral,positive,positive,positive,neutral,neutral,positive,neutral
2,Arata foarte bine 👍,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
3,bravo buna treaba,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
4,"Looking good, de vazut doar faza cu acolada de...",neutral,positive,positive,positive,positive,positive,neutral,positive,neutral,negative,positive,positive,positive,positive,neutral,positive,positive,positive,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1947,aici nu trebuie sa folosesti handleChange ci s...,neutral,neutral,neutral,negative,negative,negative,negative,neutral,neutral,neutral,negative,neutral,negative,negative,positive,neutral,neutral,neutral,neutral
1948,nu era mai simplu: sa imparti cu 20LL din prim...,neutral,neutral,neutral,neutral,negative,neutral,negative,neutral,neutral,neutral,neutral,positive,neutral,negative,negative,negative,negative,negative,negative
1949,Da asa e. Am făcut-o ca s ama prind cum merge ...,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,negative,negative
1950,"Avea un font diferit,asa am crezut ca trebuie ...",neutral,neutral,negative,negative,neutral,neutral,negative,neutral,negative,negative,neutral,neutral,negative,negative,negative,negative,neutral,negative,negative


In [55]:
# compute metrics for all model/prompt/dataset combinations
all_metrics = []

# DEVA
for ds_name, df in [('gold_standard', df_gold_standard), ('premo', df_premo)]:
    m = compute_metrics(df, ds_name, pred_col='deva_polarity', gold_col='polarity')
    m['model'] = 'DEVA'
    m['prompt'] = '—'
    all_metrics.append(m)

# LLMs
for ds_name, df in [('gold_standard', df_gold_standard_llm), ('premo', df_premo_llm)]:
    for model in MODELS:
        for prompt_id in PROMPTS:
            col = f"{model.replace(':', '_').replace('/', '_')}_prompt_{prompt_id}"
            m = compute_metrics(df, ds_name, pred_col=col, gold_col='polarity')
            m['model'] = model
            m['prompt'] = f'P{prompt_id}'
            all_metrics.append(m)

df_all_metrics = pd.concat(all_metrics, ignore_index=True)

In [56]:
# build comparison table: macro-avg P / R / F1 per tool, pivoted by dataset
macro = df_all_metrics[df_all_metrics['class'] == 'macro avg'].copy()

pivot = macro.pivot_table(
    index=['model', 'prompt'],
    columns='dataset',
    values=['precision', 'recall', 'f1']
)

# organise columns as (dataset, metric) in P / R / F1 order
pivot = pivot.swaplevel(axis=1)
metric_order = ['precision', 'recall', 'f1']
pivot = pivot[[
    (ds, m) for ds in ['gold_standard', 'premo'] for m in metric_order
]]
pivot.columns.names = ['dataset', 'metric']

# fixed row order: DEVA first, then each LLM with P0/P1/P2
row_order = [('DEVA', '\u2014')]
for model in MODELS:
    for p in range(len(PROMPTS)):
        row_order.append((model, f'P{p}'))
pivot = pivot.loc[row_order]

# style
f1_cols = [('gold_standard', 'f1'), ('premo', 'f1')]

# find best F1 overall per dataset (green)
best_overall = {col: pivot[col].max() for col in f1_cols}

# find best prompt F1 per LLM per dataset (yellow)
best_prompt = {}
for model in MODELS:
    model_rows = [(model, f'P{p}') for p in range(len(PROMPTS))]
    for col in f1_cols:
        best_val = pivot.loc[model_rows, col].max()
        for idx in model_rows:
            if pivot.loc[idx, col] == best_val:
                best_prompt[(idx, col)] = True

def highlight(data):
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for col in f1_cols:
        for idx in data.index:
            if data.loc[idx, col] == best_overall[col]:
                styles.loc[idx, col] = 'background-color: #d4edda; font-weight: bold'
            elif (idx, col) in best_prompt:
                styles.loc[idx, col] = 'background-color: #fff3cd'
    return styles

pivot.style.format('{:.3f}').apply(lambda _: highlight(pivot), axis=None)

In [57]:
# big results table:
#   rows    = (model, prompt)
#   columns = (dataset, group, metric); group in Pos/Neg/Neu/Macro/Micro, metric in P/R/F1
GROUP_LABELS   = {'positive': 'Pos', 'negative': 'Neg', 'neutral': 'Neu',
                  'macro avg': 'Macro', 'micro avg': 'Micro'}
group_order    = ['Pos', 'Neg', 'Neu', 'Macro', 'Micro']
metric_order   = ['precision', 'recall', 'f1']
metric_short   = {'precision': 'P', 'recall': 'R', 'f1': 'F1'}
dataset_order  = ['gold_standard', 'premo']
dataset_labels = {'gold_standard': 'GitHub Gold Standard', 'premo': 'PRemo'}

big = df_all_metrics[df_all_metrics['class'].isin(GROUP_LABELS)].copy()
big['group'] = big['class'].map(GROUP_LABELS)

big_pivot = big.pivot_table(
    index=['model', 'prompt'],
    columns=['dataset', 'group'],
    values=metric_order,
)

# reorder column levels (metric, dataset, group) -> (dataset, group, metric)
big_pivot = big_pivot.reorder_levels(['dataset', 'group', None], axis=1)
big_pivot.columns.names = ['dataset', 'group', 'metric']
big_pivot = big_pivot[[
    (ds, g, m)
    for ds in dataset_order
    for g in group_order
    for m in metric_order
]]
big_pivot = big_pivot.loc[row_order]

# display labels: dataset names + P/R/F1
big_pivot = big_pivot.rename(columns=dataset_labels, level='dataset').rename(columns=metric_short, level='metric')

# highlight best F1 per (dataset, group): green = best overall, yellow = best prompt per LLM
f1_cols_big = [(dataset_labels[ds], g, 'F1') for ds in dataset_order for g in group_order]
best_overall_big = {col: big_pivot[col].max() for col in f1_cols_big}
best_prompt_big = {}
for model in MODELS:
    model_rows = [(model, f'P{p}') for p in range(len(PROMPTS))]
    for col in f1_cols_big:
        best_val = big_pivot.loc[model_rows, col].max()
        for idx in model_rows:
            if big_pivot.loc[idx, col] == best_val:
                best_prompt_big[(idx, col)] = True

def highlight_big(data):
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for col in f1_cols_big:
        for idx in data.index:
            if data.loc[idx, col] == best_overall_big[col]:
                styles.loc[idx, col] = 'background-color: #d4edda; font-weight: bold'
            elif (idx, col) in best_prompt_big:
                styles.loc[idx, col] = 'background-color: #fff3cd'
    return styles

big_pivot.style.format('{:.3f}').apply(lambda _: highlight_big(big_pivot), axis=None)

## Romanian Gold Standard Results

For the Romanian dataset we run each model with three prompts in two languages: the original English prompts of Zhang et al. (`P0`, `P1`, `P2`) and their Romanian translations (`P0_ro`, `P1_ro`, `P2_ro`). This gives 18 model x prompt combinations on the same gold labels.

In [58]:
# compute metrics for all Romanian LLM runs (3 models x 6 prompts), per comment type + overall.
# df_ro_typed (built above) shares the source row order, so comment_type aligns positionally.
assert len(df_ro_gold_llm) == len(df_ro_typed), 'row count mismatch'
assert (df_ro_gold_llm['polarity'].values == df_ro_typed['polarity'].values).all(), 'row order mismatch'
df_ro_gold_llm['comment_type'] = df_ro_typed['comment_type'].values

SEGMENT_ORDER = ['review', 'issue_comment', 'review_comment', 'overall']

ro_metrics = []
for model in MODELS:
    safe_model = model.replace(':', '_').replace('/', '_')
    prompt_cols = [(f'P{p}', f'{safe_model}_prompt_{p}') for p in PROMPTS] \
                + [(f'P{p}_ro', f'{safe_model}_prompt_{p}_ro') for p in PROMPTS_RO]
    for prompt_label, col in prompt_cols:
        for seg in SEGMENT_ORDER:
            sub = df_ro_gold_llm if seg == 'overall' else df_ro_gold_llm[df_ro_gold_llm['comment_type'] == seg]
            m = compute_metrics(sub, seg, pred_col=col, gold_col='polarity')
            m['model'] = model
            m['prompt'] = prompt_label
            ro_metrics.append(m)

df_ro_metrics = pd.concat(ro_metrics, ignore_index=True).rename(columns={'dataset': 'segment'})
df_ro_metrics

,segment,class,precision,recall,f1,support,model,prompt
0,review,positive,0.602273,0.981481,0.746479,54,gemma3:12b,P0
1,review,negative,0.477273,0.700000,0.567568,30,gemma3:12b,P0
2,review,neutral,0.901639,0.504587,0.647059,109,gemma3:12b,P0
3,review,macro avg,0.660395,0.728690,0.653702,193,gemma3:12b,P0
4,review,weighted avg,0.751915,0.668394,0.662520,193,gemma3:12b,P0
...,...,...,...,...,...,...,...,...
427,overall,negative,0.199475,0.741463,0.314374,205,llama3.1:8b,P2_ro
428,overall,neutral,0.901728,0.517990,0.657998,1612,llama3.1:8b,P2_ro
429,overall,macro avg,0.476916,0.634633,0.469488,1952,llama3.1:8b,P2_ro
430,overall,weighted avg,0.788405,0.550205,0.606564,1952,llama3.1:8b,P2_ro


In [59]:
# macro-avg P/R/F1 per (model, prompt) for the Romanian dataset (overall, all comment types)
prompt_cols_ro = [f'P{p}' for p in PROMPTS] + [f'P{p}_ro' for p in PROMPTS_RO]

ro_macro = df_ro_metrics[(df_ro_metrics['class'] == 'macro avg') & (df_ro_metrics['segment'] == 'overall')].copy()
ro_macro_pivot = ro_macro.pivot_table(
    index='model',
    columns='prompt',
    values=['precision', 'recall', 'f1'],
)
ro_macro_pivot = ro_macro_pivot.reindex(MODELS)
ro_macro_pivot = ro_macro_pivot[[
    (m, p) for m in ['precision', 'recall', 'f1'] for p in prompt_cols_ro
]]
ro_macro_pivot.columns.names = ['metric', 'prompt']

# highlight best F1 per row (best prompt per model, yellow) and overall best F1 (green)
f1_cols_ro = [('f1', p) for p in prompt_cols_ro]
best_overall_ro = ro_macro_pivot[f1_cols_ro].values.max()

def highlight_ro_macro(data):
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        row_best = data.loc[idx, f1_cols_ro].max()
        for col in f1_cols_ro:
            v = data.loc[idx, col]
            if v == best_overall_ro:
                styles.loc[idx, col] = 'background-color: #d4edda; font-weight: bold'
            elif v == row_best:
                styles.loc[idx, col] = 'background-color: #fff3cd'
    return styles

ro_macro_pivot.style.format('{:.3f}').apply(lambda _: highlight_ro_macro(ro_macro_pivot), axis=None)

In [60]:
# big Romanian results table:
#   rows    = (model, prompt)
#   columns = (segment, group, metric); segment = comment type + overall,
#             group in Pos/Neg/Neu/Macro/Micro, metric in P/R/F1
ro_GROUP_LABELS = {'positive': 'Pos', 'negative': 'Neg', 'neutral': 'Neu',
                   'macro avg': 'Macro', 'micro avg': 'Micro'}
ro_group_order  = ['Pos', 'Neg', 'Neu', 'Macro', 'Micro']
ro_metric_order = ['precision', 'recall', 'f1']
ro_metric_short = {'precision': 'P', 'recall': 'R', 'f1': 'F1'}
segment_order   = ['review', 'issue_comment', 'review_comment', 'overall']
segment_labels  = {'review': 'Review', 'issue_comment': 'Issue Comment',
                   'review_comment': 'Review Comment', 'overall': 'Overall'}
prompt_cols_ro  = [f'P{p}' for p in PROMPTS] + [f'P{p}_ro' for p in PROMPTS_RO]
row_order_ro    = [(m, p) for m in MODELS for p in prompt_cols_ro]

ro_big = df_ro_metrics[df_ro_metrics['class'].isin(ro_GROUP_LABELS)].copy()
ro_big['group'] = ro_big['class'].map(ro_GROUP_LABELS)

ro_big_pivot = ro_big.pivot_table(
    index=['model', 'prompt'],
    columns=['segment', 'group'],
    values=ro_metric_order,
)

# reorder column levels (metric, segment, group) -> (segment, group, metric)
ro_big_pivot = ro_big_pivot.reorder_levels(['segment', 'group', None], axis=1)
ro_big_pivot.columns.names = ['segment', 'group', 'metric']
ro_big_pivot = ro_big_pivot[[
    (seg, g, m)
    for seg in segment_order
    for g in ro_group_order
    for m in ro_metric_order
]]
ro_big_pivot = ro_big_pivot.loc[row_order_ro]

# display labels: segment names + P/R/F1
ro_big_pivot = ro_big_pivot.rename(columns=segment_labels, level='segment').rename(columns=ro_metric_short, level='metric')

# highlight best F1 per (segment, group): green = best overall, yellow = best prompt per model
f1_cols_ro_big = [(segment_labels[seg], g, 'F1') for seg in segment_order for g in ro_group_order]
best_overall_ro_big = {col: ro_big_pivot[col].max() for col in f1_cols_ro_big}
best_prompt_ro_big = {}
for model in MODELS:
    model_rows = [(model, p) for p in prompt_cols_ro]
    for col in f1_cols_ro_big:
        best_val = ro_big_pivot.loc[model_rows, col].max()
        for idx in model_rows:
            if ro_big_pivot.loc[idx, col] == best_val:
                best_prompt_ro_big[(idx, col)] = True

def highlight_ro_big(data):
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for col in f1_cols_ro_big:
        for idx in data.index:
            if data.loc[idx, col] == best_overall_ro_big[col]:
                styles.loc[idx, col] = 'background-color: #d4edda; font-weight: bold'
            elif (idx, col) in best_prompt_ro_big:
                styles.loc[idx, col] = 'background-color: #fff3cd'
    return styles

ro_big_pivot.style.format('{:.3f}').apply(lambda _: highlight_ro_big(ro_big_pivot), axis=None)

## Prompt sensitivity

LLM predictions are sensitive to the exact wording of the prompt even when the task definition is fixed. The three tables below quantify this from different angles:

- **Macro F1 spread**: for each (model, dataset) pair, the gap between the best and worst prompt is the sensitivity score. A range above ~0.05 macro F1 is a clear sensitivity signal.
- **Predicted class distribution**: shows that prompts shift not just accuracy but *what the model predicts*. Terse prompts default to neutral; verbose prompts inflate non-neutral predictions.
- **Prompt-language effect** (Romanian dataset only): compares English-language versus Romanian-language prompts at otherwise identical wording.


In [61]:
# macro F1 per (model, dataset, prompt)

_t1_specs = [
    ('gold_standard',    df_gold_standard_llm, [(p, '')    for p in PROMPTS]),
    ('premo',            df_premo_llm,         [(p, '')    for p in PROMPTS]),
    ('ro_gold_standard', df_ro_gold_llm,       [(p, '')    for p in PROMPTS] +
                                               [(p, '_ro') for p in PROMPTS_RO]),
]

_t1_rows = []
for ds_name, df, prompt_specs in _t1_specs:
    for model in MODELS:
        safe = model.replace(':', '_').replace('/', '_')
        for pid, suf in prompt_specs:
            col = f'{safe}_prompt_{pid}{suf}'
            m = compute_metrics(df, ds_name, pred_col=col)
            f1 = float(m.loc[m['class'] == 'macro avg', 'f1'].iloc[0])
            _t1_rows.append({
                'model':   model,
                'dataset': ds_name,
                'prompt':  f'P{pid}{suf}',
                'f1':      f1,
            })

_t1 = pd.DataFrame(_t1_rows)

# split the Romanian dataset by prompt language so each row averages over a coherent prompt set
def _group(row):
    if row['dataset'] == 'ro_gold_standard':
        return 'ro_gold_standard (RO prompts)' if row['prompt'].endswith('_ro') else 'ro_gold_standard (EN prompts)'
    return row['dataset']
_t1['group']     = _t1.apply(_group, axis=1)
_t1['prompt_id'] = _t1['prompt'].str.replace('_ro', '', regex=False)

t1_pivot = _t1.pivot_table(index=['model', 'group'], columns='prompt_id', values='f1')[['P0', 'P1', 'P2']]
t1_pivot['min']   = t1_pivot[['P0', 'P1', 'P2']].min(axis=1)
t1_pivot['max']   = t1_pivot[['P0', 'P1', 'P2']].max(axis=1)
t1_pivot['range'] = t1_pivot['max'] - t1_pivot['min']
t1_pivot['best']  = t1_pivot[['P0', 'P1', 'P2']].idxmax(axis=1)
t1_pivot['worst'] = t1_pivot[['P0', 'P1', 'P2']].idxmin(axis=1)

t1_pivot.style.format({c: '{:.3f}' for c in ['P0', 'P1', 'P2', 'min', 'max', 'range']}) \
              .background_gradient(subset=['range'], cmap='Reds')


In [62]:
# prompt-language effect on the Romanian dataset
ro_macro = df_ro_metrics[df_ro_metrics['class'] == 'macro avg'].copy()
ro_macro['lang'] = ro_macro['prompt'].apply(lambda p: 'RO' if p.endswith('_ro') else 'EN')

lang_pivot = ro_macro.pivot_table(
    index='model',
    columns='lang',
    values='f1',
    aggfunc='mean',
)
lang_pivot['delta (RO - EN)'] = lang_pivot['RO'] - lang_pivot['EN']
lang_pivot.style.format('{:+.3f}', subset=['delta (RO - EN)']) \
                .format('{:.3f}', subset=['EN', 'RO'])


lang,EN,RO,delta (RO - EN)
model,,,
gemma3:12b,0.570,0.606,+0.035
llama3.1:8b,0.384,0.510,+0.125
phi4:14b,0.480,0.462,-0.018


## Parser source and fallback rate

A single unified parser is used for all LLM responses, regardless of whether the prompt was in English or in Romanian. For each cache file, count how many labels each parser stage produced. Stages, in decreasing order of confidence:

- `marker` -- explicit `Label:` / `Sentiment:` / `Etichetă:` declaration
- `bold` -- markdown bold declaration `**Negativ**` or `**(2) neutral**`
- `numbered` -- numeric prefix attached to a label word, e.g. `(2) neutral`
- `bare_numbered` -- response starts with `(N)` alone with no label word; the prompt convention defines (1)=positive, (2)=neutral, (3)=negative
- `token` -- single-token substring match (used when no structured declaration is found)
- `default` -- nothing matched anywhere; the parser conservatively returned `neutral`

The **fallback** column reports `default` as a share of total entries. A high fallback rate is a red flag: it means the model produced output that did not contain any of the canonical label words and did not start with a recognisable numeric option, so the prediction is essentially uninformed.


In [63]:
# Cache files are named "{model_safe}_prompt_{N|N_ro}_{dataset}.json".
# The `(?:_ro)??` is LAZY so that `prompt_0_ro_gold_standard.json` is parsed
# as (prompt=0, dataset=ro_gold_standard) rather than (prompt=0_ro,
# dataset=gold_standard). The dataset alternation lists the longest match
# first to remove any residual ambiguity.
_FNAME_RE = _re.compile(
    r'(?P<model>.+?)_prompt_(?P<prompt>\d+(?:_ro)??)_'
    r'(?P<dataset>ro_gold_standard|gold_standard|premo)\.json$'
)
_MODEL_DISPLAY = {m.replace(':', '_').replace('/', '_'): m for m in MODELS}

source_rows = []
for path in sorted(CACHE_DIR.glob('*.json')):
    m = _FNAME_RE.match(path.name)
    if not m:
        continue

    entries = json.loads(path.read_text(encoding='utf-8'))
    counts = Counter()
    for e in entries:
        raw = e.get('raw_message') or ''
        if not raw:
            counts['default'] += 1
            continue
        _, src = extract_label_with_source(raw)
        counts[src] += 1

    source_rows.append({
        'dataset':       m['dataset'],
        'model':         _MODEL_DISPLAY.get(m['model'], m['model']),
        'prompt':        f"P{m['prompt']}",
        'n':             len(entries),
        'marker':        counts.get('marker', 0),
        'bold':          counts.get('bold', 0),
        'numbered':      counts.get('numbered', 0),
        'bare_numbered': counts.get('bare_numbered', 0),
        'token':         counts.get('token', 0),
        'default':       counts.get('default', 0),
    })

df_source = pd.DataFrame(source_rows)

df_source_display = df_source.copy()
for col in ['marker', 'bold', 'numbered', 'bare_numbered', 'token', 'default']:
    df_source_display[col] = (df_source_display[col] / df_source_display['n']).map('{:.1%}'.format)

df_source_display = df_source_display.sort_values(
    ['dataset', 'model', 'prompt']
).reset_index(drop=True)

df_source_display


,dataset,model,prompt,n,marker,bold,numbered,bare_numbered,token,default
0,gold_standard,gemma3:12b,P0,2137,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%
1,gold_standard,gemma3:12b,P1,2137,76.0%,16.0%,7.9%,0.0%,0.2%,0.0%
2,gold_standard,gemma3:12b,P2,2137,8.6%,0.0%,91.3%,0.0%,0.0%,0.1%
3,gold_standard,llama3.1:8b,P0,2137,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%
4,gold_standard,llama3.1:8b,P1,2137,0.0%,0.0%,99.8%,0.0%,0.0%,0.1%
5,gold_standard,llama3.1:8b,P2,2137,0.2%,0.0%,99.3%,0.0%,0.1%,0.5%
6,gold_standard,phi4:14b,P0,2137,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%
7,gold_standard,phi4:14b,P1,2137,6.6%,39.1%,51.0%,0.0%,3.4%,0.0%
8,gold_standard,phi4:14b,P2,2137,2.4%,39.3%,48.9%,0.0%,9.4%,0.0%
9,premo,gemma3:12b,P0,1775,0.0%,0.0%,0.0%,0.0%,100.0%,0.0%


# Export combined predictions for downstream analysis
Writes one parquet per dataset to `output/predictions/`.

In [64]:
def build_predictions(df_llm, deva_out_file=None):
    out = df_llm.copy()
    if deva_out_file is not None:
        v, a, lab = parse_deva_output(deva_out_file)
        assert len(v) == len(out), f"DEVA length mismatch: {len(v)} vs {len(out)}"
        out['deva_valence']  = v
        out['deva_arousal']  = a
        out['deva_label']    = lab
        out['deva_polarity'] = out['deva_valence'].apply(valence_to_polarity)
    return out

pred_gs = build_predictions(df_gold_standard_llm, DEVA_OUTPUT_DIR / "gold_standard_out.txt")
pred_pr = build_predictions(df_premo_llm,         DEVA_OUTPUT_DIR / "premo_out.txt")
pred_ro = build_predictions(df_ro_gold_llm)  # no DEVA for Romanian

pred_gs.to_parquet(PRED_DIR / "predictions_gold_standard.parquet")
pred_pr.to_parquet(PRED_DIR / "predictions_premo.parquet")
pred_ro.to_parquet(PRED_DIR / "predictions_ro_gold_standard.parquet")

print(f"Wrote {PRED_DIR / 'predictions_gold_standard.parquet'}: {pred_gs.shape}")
print(f"Wrote {PRED_DIR / 'predictions_premo.parquet'}: {pred_pr.shape}")
print(f"Wrote {PRED_DIR / 'predictions_ro_gold_standard.parquet'}: {pred_ro.shape}")
pred_gs.head(3)


Wrote /home/davidm/University/Year 3 Resources/BEP/repo/output/predictions/predictions_gold_standard.parquet: (2137, 15)
Wrote /home/davidm/University/Year 3 Resources/BEP/repo/output/predictions/predictions_premo.parquet: (1791, 15)
Wrote /home/davidm/University/Year 3 Resources/BEP/repo/output/predictions/predictions_ro_gold_standard.parquet: (1952, 21)


,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2,deva_valence,deva_arousal,deva_label,deva_polarity
0,Guess there is a typo here: `translate`.,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,negative,neutral,1,-3,NEUTRAL (LA),neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,neutral,neutral,neutral,neutral,negative,negative,negative,neutral,negative,4,-3,RELAXED,positive
2,Thank you Vlad. Your contribution to Mangos an...,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive,3,-4,RELAXED,positive
